# Graph coloring: five phase-oracle synthesis methods

Compare **Specialized XAG**, **AIG + Bennett**, **XAG + Bennett**, **4-LUT + Bennett**, and **Best-fit LHRS** on the same coloring functions. The K3 walkthrough follows each method from logic network to quantum phase marking. The final experiment covers the nine paper graphs.

**Scope:** the complete phase-flip oracle, including uncomputation. Width means **total qubits**. Depth and gate count are reported twice: as synthesized and after decomposition into **U/CX**, without a hardware coupling map. State preparation, Grover diffusion, measurement, and hardware execution are excluded. These new oracle-only counts are not reproductions of the historical full-Grover numbers.

## 1. Reproducible setup

Select the repository's Python 3.10 `.venv` kernel. Install the research extras and system Graphviz as described in [setup](../../docs/setup.md). The notebook does not install packages or build dependencies.

For a fresh native build using the pinned local Caterpillar checkout, run these commands from the repository root **before** executing the notebook:

```bash
cmake -S external/native/boolean_synthesis -B external/native/boolean_synthesis/build/phase-oracle \
  -DCMAKE_BUILD_TYPE=Release -DFETCHCONTENT_SOURCE_DIR_CATERPILLAR="$PWD/external/caterpillar"
cmake --build external/native/boolean_synthesis/build/phase-oracle -j 2
export LOGIQUE_NATIVE_EXECUTABLE="$PWD/external/native/boolean_synthesis/build/phase-oracle/boolean_synthesis"
uv run --no-sync logique notebook run experiments/studies/coloring_phase_oracle_comparison.ipynb --offline --timeout 3600
```

This study retains Caterpillar's bundled Mockturtle and Kitty and the bridge's two documented cleanup fixes. The separate top-level Mockturtle and ABC checkouts do not change this experiment. Record effective build provenance, not just the list of checked-out submodules.

The standard configuration fixes LUT size 4, best-fit outer/inner cuts 16/4, and seed 7. All figures and circuit files go to a fresh directory under `dump/`; committed source outputs stay empty. For a quick smoke run, set `LOGIQUE_PHASE_STUDY_SMOKE=1` before launching the kernel. `LOGIQUE_PHASE_RESULTS` may point to a completed run with the identical configuration and executable to reuse it for the final charts.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
from uuid import uuid4
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display, Markdown
from qiskit import qpy
from logique import read_dimacs, encode_coloring
from logique.paths import native_executable
from logique.synthesis.result import SynthesisResult
from logique.benchmarks.runner import load_config, run_benchmarks, new_run_directory
from logique.benchmarks.results import load_results
from logique.benchmarks.provenance import environment, checkout_revision
from logique.visualization.phase import METHOD_LABELS, phase_walkthrough, phase_charts

if "LOGIQUE_WORKSPACE" in os.environ:
    ROOT = Path(os.environ["LOGIQUE_WORKSPACE"]).expanduser().resolve()
else:
    current = Path.cwd().resolve()
    ROOT = next(
        p
        for p in (current, *current.parents)
        if (p / "pyproject.toml").exists() and (p / "src/logique").is_dir()
    )
if "LOGIQUE_NATIVE_EXECUTABLE" in os.environ:
    EXECUTABLE = native_executable()
else:
    candidates = [
        ROOT / "external/native/boolean_synthesis/build/phase-oracle/boolean_synthesis",
        ROOT / "external/native/boolean_synthesis/build/boolean_synthesis",
    ]
    EXECUTABLE = native_executable(next((p for p in candidates if p.is_file()), None))
if shutil.which("dot") is None:
    raise RuntimeError("Install system Graphviz (dot) before running the illustrations")
OUT = (
    Path(os.environ["LOGIQUE_OUTPUT"]).resolve()
    if "LOGIQUE_OUTPUT" in os.environ
    else new_run_directory(ROOT / "dump" / uuid4().hex)
)
OUT.mkdir(parents=True, exist_ok=True)
CONFIG = load_config(ROOT / "configs/coloring_phase_oracle_comparison.json")
SMOKE = os.environ.get("LOGIQUE_PHASE_STUDY_SMOKE") == "1"
if SMOKE:
    CONFIG["inputs"] = [p for p in CONFIG["inputs"] if Path(p).stem == "K3"]
provenance = {
    "environment": environment(),
    "checkout": checkout_revision(ROOT),
    "native_executable": str(EXECUTABLE),
    "native_sha256": hashlib.sha256(EXECUTABLE.read_bytes()).hexdigest(),
    "config": CONFIG,
    "smoke": SMOKE,
}
build_info = EXECUTABLE.parent / "build_provenance.json"
provenance["native_build"] = (
    json.loads(build_info.read_text()) if build_info.exists() else None
)
(OUT / "study.json").write_text(json.dumps(provenance, indent=2) + "\n")
display(
    pd.DataFrame(
        [{"method": METHOD_LABELS[m], "backend": m} for m in CONFIG["methods"]]
    )
)
display(provenance)
print("Artifacts:", OUT)


## 2. K3: what the oracle marks

Each vertex uses two input bits, least-significant bit first. Codes 00, 01, and 10 represent colors 0, 1, and 2; code 11 is invalid. The benchmark fixes vertex 1 to color 0. Its full predicate is

\[
f(x)=[c_1=0]\land\bigwedge_{v=1}^{3}[c_v<3]\land[c_1\ne c_2]\land[c_1\ne c_3]\land[c_2\ne c_3].
\]

There are **two satisfying assignments**: (0,1,2) and (0,2,1). The oracle must leave all other input assignments unmarked, including invalid codes and precolor violations. Fixed vertices remain explicit input wires; their constraints are checked by the predicate.

For each raw computation F, the wrapper applies **F, Z on the computed output, then F†**. The final action is

\[
O_f|x\rangle|0\cdots0\rangle=(-1)^{f(x)}|x\rangle|0\cdots0\rangle.
\]

The computed output is an auxiliary wire restored to zero; no extra phase-kickback target is appended. All 64 input states are checked by tracking their permutation and complex phase through the actual logical Qiskit circuit. Linearity then establishes the same action on superpositions with clean initial workspace. This avoids allocating a statevector over up to 40 K3 circuit wires.

In [ ]:
problem = read_dimacs(ROOT / "datasets/graphs/benchmarks/K3.col")
encoding = encode_coloring(problem)
(OUT / "K3_predicate.v").write_text(encoding.verilog())
fig, ax = plt.subplots(figsize=(5, 3.5), constrained_layout=True)
nx.draw_networkx(
    problem.to_networkx(),
    pos={1: (0, 1), 2: (-1, 0), 3: (1, 0)},
    labels={1: "v1\ncolor 0", 2: "v2", 3: "v3"},
    node_color=["#e9c879", "#c7d9ef", "#c7d9ef"],
    node_size=1800,
    ax=ax,
    font_size=11,
)
ax.set_title("K3 · three colors · vertex 1 fixed")
ax.axis("off")
for ext in ("svg", "png"):
    fig.savefig(OUT / f"K3_graph.{ext}", dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)
display(
    pd.DataFrame(
        [
            {
                "vertex": v,
                "input bits (LSB first)": qs,
                "precolor": problem.precolored.get(v, "free"),
            }
            for v, qs in encoding.input_mapping.items()
        ]
    )
)
k3_config = {
    **CONFIG,
    "inputs": [str(ROOT / "datasets/graphs/benchmarks/K3.col")],
    "output": str(OUT / "k3"),
}
K3_RUN = run_benchmarks(k3_config, executable=EXECUTABLE)
k3_manifest = json.loads((K3_RUN / "run.json").read_text())
K3_RESULTS = {}
for case in k3_manifest["cases"]:
    with (K3_RUN / f"{case['name']}_raw.qpy").open("rb") as stream:
        raw = qpy.load(stream)[0]
    metadata = json.loads((K3_RUN / f"{case['name']}.json").read_text())
    K3_RESULTS[case["method"]] = SynthesisResult(raw, metadata)
assert all(
    r.metadata["phase_validation"]["marked_assignments"] == 2
    for r in K3_RESULTS.values()
)
truth = pd.DataFrame(
    [
        {
            "basis": x,
            "bits x5…x0": format(x, "06b"),
            "colors (v1,v2,v3)": tuple(encoding.decode(x).values()),
            "verified phase": -1 if problem.is_valid(encoding.decode(x)) else 1,
        }
        for x in range(64)
    ]
)
truth.to_csv(OUT / "K3_phase_truth.csv", index=False)
display(Markdown("**Marked assignments, verified by all five methods:**"))
display(truth[truth["verified phase"] == -1])
display(Markdown("**Complete verified input table:**"))
with pd.option_context("display.max_rows", 64):
    display(truth)


## 3. Method walkthroughs

These are five configured compilation paths, not five independent inventions. AIG + Bennett uses the imported structure converted to AIG. The XAG paths use the bridge's refactored XAG; 4-LUT and best-fit start from that XAG. The comparison therefore includes these preprocessing choices. It does not isolate representation alone.

Logical counts retain each controlled operation as one gate, including a multi-controlled gate. LUT blocks are expanded through the installed Tweedledum PPRM synthesis before counting. The U/CX charts expose the additional decomposition cost. No hypothetical logarithmic-depth weight is substituted for an actual decomposition.

### 3.1 Specialized XAG

The refactored graph preserves explicit XOR operations. Specialized synthesis exploits that structure and uses phase-sensitive controlled rotations for temporary AND computations. The raw computation F may leave scratch values or phases; F† removes them after the output Z. Read the full circuit to see those gates, rather than treating every controlled rotation as an exact Toffoli.

In [ ]:
phase_walkthrough(K3_RESULTS["xag"], OUT / "illustrations")


### 3.2 AIG + Bennett

An AIG uses two-input AND nodes and complemented edges. XOR relationships must be expressed in this representation. Bennett mapping computes intermediate nodes in dependency order and reverses intermediate computation. The exported schedule describes F; the outer phase wrapper then marks its output and reverses all of F. This implementation retains the standard wrapper even if more specialized phase-only cleanup could remove redundant work.

In [ ]:
phase_walkthrough(K3_RESULTS["aig_bennett"], OUT / "illustrations")


### 3.3 XAG + Bennett

The Bennett strategy operates on the refactored XOR/AND network. Compare its network with Specialized XAG to separate the common representation from the different quantum mapping. Compare its retained intermediates and circuit depth with AIG + Bennett. Equal logical gate counts do not imply equal decomposed costs.

In [ ]:
phase_walkthrough(K3_RESULTS["xag_bennett"], OUT / "illustrations")


### 3.4 4-LUT + Bennett

The refactored XAG is covered by cells with at most four inputs. Each cell's truth table defines a local Boolean function, which PPRM synthesis expands into reversible gates. Bennett controls the compute/uncompute schedule. Four is the **LUT input bound**, not the number of graph colors. Fewer logic nodes can still produce more quantum gates.

In [ ]:
phase_walkthrough(K3_RESULTS["klut_bennett"], OUT / "illustrations")


### 3.5 Best-fit LHRS

Best-fit starts with an outer LUT covering, follows an eager cleanup strategy, and remaps each cell using an inner LUT size that fits the available clean ancillae. Here outer/inner limits are 16/4. The graph shown is the underlying XAG; the accompanying actual cell cuts and emitted LUT operations describe the mapping. The compact K3 result illustrates a workspace/decomposition tradeoff, not a general claim that this method wins on every graph.

In [ ]:
phase_walkthrough(K3_RESULTS["best_fit"], OUT / "illustrations")


## 4. K3 resource comparison

Both views measure F–Z–F†. **Logical:** as emitted, with LUTs expanded but controlled gates retained. **Decomposed:** Qiskit lowering to U/CX, optimization level 1, seed 7, no routing. U denotes an arbitrary one-qubit gate; this cost model is not a T-count estimate. Auxiliary qubits include both the computed output and scratch. Width counts all allocated quantum wires, including any idle wires.

In [ ]:
k3_metrics = load_results(K3_RUN)
display(
    k3_metrics[
        [
            "method",
            "metric_level",
            "depth",
            "qubits",
            "gates",
            "inputs",
            "auxiliary_qubits",
            "validation",
        ]
    ]
)
phase_charts(K3_RUN, OUT / "k3_charts")


## 5. Nine paper graphs

Run the exact same five paths on **K3, K5, K10, K20, myciel3–5, 2-Insertions_3, and zed-city_2019**. Each input supplies its color count and precolors. All methods see the same complete predicate.

Validation is exhaustive up to 12 input bits and uses 256 deterministic samples otherwise. **Sampled validation is not a correctness proof**; the report includes how many sampled assignments were marked, since satisfying colorings can be rare. Validation compares the Boolean networks and quantum mapping, then checks phase action against the independent graph predicate. U/CX lowering is separately cross-checked on small regression cases; large lowered circuits are not exhaustively simulated.

Runs checkpoint completed cases. A failure keeps diagnostics and completed rows, raises an error, and prevents a partial table from being presented as a complete comparison. Full execution produces 45 cases and 90 metric rows. Smoke execution intentionally produces five cases and ten rows. Charts use a labeled logarithmic y-axis when values span at least 100×, so smaller methods remain visible; narrower ranges use a linear axis.

In [ ]:
reuse = os.environ.get("LOGIQUE_PHASE_RESULTS")
if reuse:
    RUN = Path(reuse).expanduser().resolve()
    manifest = json.loads((RUN / "run.json").read_text())
    if manifest["status"] != "complete":
        raise ValueError("Only a completed phase benchmark run can be reused")
    for key, value in CONFIG.items():
        if key != "output" and manifest["config"].get(key) != value:
            raise ValueError(f"Reused run differs in configuration field {key}")
    if manifest["native"]["sha256"] != provenance["native_sha256"]:
        raise ValueError("Reused run was generated by a different native executable")
    for case in manifest["cases"]:
        if (
            hashlib.sha256(Path(case["input"]).read_bytes()).hexdigest()
            != case["sha256"]
        ):
            raise ValueError("Benchmark source changed since the reused run")
else:
    RUN = run_benchmarks(
        {**CONFIG, "output": str(OUT / "benchmark")}, executable=EXECUTABLE
    )
    manifest = json.loads((RUN / "run.json").read_text())
metrics = load_results(RUN)
expected_cases = len(CONFIG["inputs"]) * len(CONFIG["methods"])
assert len(manifest["cases"]) == expected_cases
assert all(case["status"] == "complete" for case in manifest["cases"])
assert len(metrics) == 2 * expected_cases
assert not metrics.duplicated(["benchmark", "method", "metric_level"]).any()
provenance["benchmark_run"] = str(RUN)
provenance["benchmark_manifest_sha256"] = hashlib.sha256(
    (RUN / "run.json").read_bytes()
).hexdigest()
(OUT / "study.json").write_text(json.dumps(provenance, indent=2) + "\n")
metrics.to_csv(OUT / "phase_oracle_metrics.csv", index=False)
display(
    metrics[
        [
            "benchmark",
            "method",
            "metric_level",
            "qubits",
            "depth",
            "gates",
            "validation",
            "marked_samples",
        ]
    ]
)
phase_charts(RUN, OUT / "benchmark_charts")


## 6. Resource ratios and interpretation

Each ratio is **method cost / Specialized XAG cost** for the same graph and metric level. Values below 1 indicate a lower measured cost. Width is total allocated qubits; gate count and depth are properties of the complete oracle at the stated gate basis. A classical logic-depth reduction need not reduce quantum depth after scheduling and decomposition.

In [ ]:
ratio_rows = []
for level in ("logical", "decomposed"):
    frame = metrics[metrics.metric_level == level]
    baseline = frame[frame.method == "xag"].set_index("benchmark")
    for row in frame.to_dict("records"):
        ratio_rows.append(
            {
                "benchmark": row["benchmark"],
                "method": METHOD_LABELS[row["method"]],
                "metric_level": level,
                **{
                    f"relative_{metric}": row[metric]
                    / baseline.loc[row["benchmark"], metric]
                    for metric in ("depth", "qubits", "gates")
                },
            }
        )
ratios = pd.DataFrame(ratio_rows)
ratios.to_csv(OUT / "ratios_to_specialized.csv", index=False)
display(ratios.round(3))
observations = []
for (benchmark, level), frame in metrics.groupby(
    ["benchmark", "metric_level"], sort=False
):
    observations.append(
        {
            "benchmark": benchmark,
            "metric_level": level,
            **{
                f"lowest_{metric}": ", ".join(
                    METHOD_LABELS[m]
                    for m in frame.loc[frame[metric] == frame[metric].min(), "method"]
                )
                for metric in ("depth", "qubits", "gates")
            },
        }
    )
display(pd.DataFrame(observations))
print("Saved phase circuits and metadata:", RUN)
print("Saved notebook figures, metrics, ratios, and provenance:", OUT)


The tables above report outcomes for these pinned implementations and parameters. They include the generic phase wrapper's cost and do not claim optimal phase-oracle synthesis. A separate study could sweep LUT sizes, optimize phase-only cleanup, or introduce new logic balancing passes; none changes this baseline experiment.

References: [Caterpillar mapping strategies](https://qcaterpillar.readthedocs.io/en/latest/strategies.html), [LHRS](https://www.microsoft.com/en-us/research/publication/hierarchical-reversible-logic-synthesis-using-luts-2/), and the repository's [native bridge notes](../../external/native/boolean_synthesis/README.md).